<div align="right" style="text-align: right"><i>Peter Norvig<br>Sept 2026</i></div>

# Integer Palindromes

Consider this problem:

***Write a function to return the integer [palindromic numbers](https://en.wikipedia.org/wiki/Palindromic_number) written in base 10, with no zero digits, and whose digits add up to N.***


(This is a generalization of a problem from the [2026 AIME](https://artofproblemsolving.com/wiki/index.php/2026_AIME_I_Problems/Problem_2?srsltid=AfmBOor4WAOSBdOAsLKKfku51DG21_h8NUzWEenLIieAMaUFWilw8QeB) (American Invitational Mathematics Examination) which asked only for the count of palindromes with digit sum *N* = 13.) 

Here are my thoughts about the problem:
- A palindrome is a sequence of characters. It is the base 10 string of characters, not the integer itself, that is a palindrome.
  - Thus, my function's signature will be `palindromes_with_digit_sum(N: int) -> list[str]`
  - For example, `palindromes_with_digit_sum(4) == ['1111',  '121', '22',  '4']`.
  - The empty string is [generally considered](https://en.wikipedia.org/wiki/Empty_string) a palindrome.
- Valid palindromes can be classified into two kinds:
  - Base cases: a valid palindrome can be the empty string (when *N* = 0) or can be a single digit (when 1 ≤ *N* ≤ 9).
  - Multi-digit cases: a valid palindrome can have first and last digit *d*, and a middle part that is a valid palindrome.
    - Middle parts can be found by recursively asking for valid palindromes with digit sum equal to *N* - 2*d*.

# Solution

Those thoughts easily translate into a function that is concise and reasonably efficient:

In [1]:
from functools import cache

@cache
def palindromes_with_digit_sum(N: int) -> list[str]:
    """Palindromic digit-strings, with no zero digits, whose digits sum to `N`."""
    base_cases  = [''] if N == 0 else [str(N)] if 1 <= N <= 9 else [] # 0 or 1 digits
    multi_cases = [f'{d}{middle}{d}'                                  # 2 or more digits
                  for d in range(1, 10) if d * 2 <= N
                  for middle in palindromes_with_digit_sum(N - 2 * d)]
    return multi_cases + base_cases

Here's the program in action:

In [2]:
palindromes_with_digit_sum(7)

['1111111', '11311', '12121', '151', '21112', '232', '313', '7']

That looks right.

# TL;DR: Comparing Human and LLM Solutions

I asked four Large Language Model coding agents (Gemini 3.1 Pro, GPT 5.6 Sol, Claude Sonnet 5, and Kimi K3) to solve this problem.

On the **positive** side, they all wrote correct code that passed all my tests, and they all got the idea that the first part of the digit string has to mirror the last part,  and that you shouldn't generate half-parts with a sum greater than *N*/2.

On the **negative** side, they all ran 10 to 20 times slower  compared to my solution (on the particular test suite), largely because none of them used `@cache`. Only Gemini produced what I considered a clear, simple solution (basically the same as mine, but using statements instead of expressions). All the others had overly complicated code (with a line count 6 times mine). Only GPT provided type hints.

Looking at the code they wrote, it seems like they all failed to realize that the size of the result increases exponentially as roughly 2<sup>*N*/2</sup>, and that caching could reduce an exponential number of calls.  All but Gemini failed to realize that there is a simple recursive rule that determines the results for *N* in terms of the results for *N-2* through *N*-18. So I decided to give each LLM a second chance, this time using a prompt that contains two hints:

>*Write a function to return the integer palindromic numbers written in base 10, with no zero digits, and whose digits add up to N.*
>
>*Keep in mind these two very general CS ideas:*
>
>1. *Functions that are called repeatedly with the same argument can often benefit from caching.*
>
>2. *A problem with parameter N can often be broken down to a call or calls to the same function with smaller parameter values, followed by some adaptation to combine the smaller result(s) into the final result.*


For each coding agent, the table below lists the lines of code (including blank lines and comments), the run time in seconds on the test suite, and my subjective code style judgment. For the non-human agents, there are also columns for their second prompt (incorporating the two hints). The hints helped a lot! That is a bit surprising: surely the models in some sense "knew" the two hints – they had seen lots of example problems where each of the hints were applied – but they failed to properly apply what they "knew" without the hint. So (at least this time) the human programmer  came out on top.

|Author|Lines 1|Time 1|Style 1|Lines 2|Time 2|Style 2|
|-----|---:|---:|:---:|---:|---:|:---:|
|Human  |10| 0.7|Good|– |–  |–   |
|Gemini |19| 7.5|Good|42|2.5|Okay|
|GPT    |60|10.1|Bad |25|1.1|Good|
|Claude |62|16.4|Bad |20|1.9|Good|
|Kimi   |62|14.6|Bad |31|1.0|Good|

Interestingly, Gemini, which had the best and most concise code with the first prompt, ended up with the least-good and most verbose code after the hints (the hint doubled the lines of code for Gemini and cut it in half for the others).

# Analysis

Here are some facts I discovered while writing my code:
   - If *N* is odd, each palindrome must have an odd digit-length, with an odd middle digit.
   - If *N* is even, each palindrome can have an even length, or an odd length with an even middle digit.
   - The number of solutions for *N* = 2*m* is the same as for *N* = 2*m* + 1. Here's why:
      - For each even-length 2*m* solution, you can insert 1 as the middle digit.
      - For each odd-length 2*m* solution, you can increase the (even) middle digit by 1.
   - The number of solutions for *N* + 2 is either double or slightly less than double the number of solutions for *N*.
      - The doubling breaks down when we run out of digits. For example, with *N* = 7, one solution is "7", and for *N* + 2 we can change that solution to either "171" or "9", doubling the count (and similarly for every other palindrome with digit sum *N* = 7). But for the case when *N* = 8, one solution is "8", and we can get "181", but we can't add 2 to the digit "8", so we get one less than double the number of solutions.

Exploring the counts of number of valid palindromes, and whether they double when *N* increases by 2:

In [3]:
def count(N) -> int: return len(palindromes_with_digit_sum(N))

# Does the count double when N increases by 2?
for N in range(22):
    print(f'count({N+2:2})/count({N:2}) = {count(N+2):4}/{count(N):<4} '
          f'= {count(N+2) / count(N):.3f}')

count( 2)/count( 0) =    2/1    = 2.000
count( 3)/count( 1) =    2/1    = 2.000
count( 4)/count( 2) =    4/2    = 2.000
count( 5)/count( 3) =    4/2    = 2.000
count( 6)/count( 4) =    8/4    = 2.000
count( 7)/count( 5) =    8/4    = 2.000
count( 8)/count( 6) =   16/8    = 2.000
count( 9)/count( 7) =   16/8    = 2.000
count(10)/count( 8) =   31/16   = 1.938
count(11)/count( 9) =   31/16   = 1.938
count(12)/count(10) =   62/31   = 2.000
count(13)/count(11) =   62/31   = 2.000
count(14)/count(12) =  124/62   = 2.000
count(15)/count(13) =  124/62   = 2.000
count(16)/count(14) =  248/124  = 2.000
count(17)/count(15) =  248/124  = 2.000
count(18)/count(16) =  496/248  = 2.000
count(19)/count(17) =  496/248  = 2.000
count(20)/count(18) =  991/496  = 1.998
count(21)/count(19) =  991/496  = 1.998
count(22)/count(20) = 1980/991  = 1.998
count(23)/count(21) = 1980/991  = 1.998


Below are the solutions found by all four coding agents, for both prompts. They each chose their own function names, but I appended the model name so we can keep track, and included a `_2_` in the name of functions that came from the second prompt.

# Gemini 

For the first prompt:

In [4]:
def gemini_generate_palindromes(target_sum):
    """
    Yields all positive integer palindromes (as strings) with no zero digits
    whose digits add up to the target_sum.
    """
    # Base case 1: The remaining sum is exactly 0 (even-length palindrome center)
    if target_sum == 0:
        yield ""
        
    # Base case 2: The remaining sum is a single non-zero digit (odd-length center)
    if 1 <= target_sum <= 9:
        yield str(target_sum)
        
    # Recursive step: Wrap matching pairs of digits (1 through 9) around the outside
    for d in range(1, 10):
        # We need enough remaining sum for two 'd' digits
        if target_sum >= 2 * d:
            for inner in gemini_generate_palindromes(target_sum - 2 * d):
                yield f"{d}{inner}{d}"

For the second prompt, with the two hints:

In [5]:
from functools import lru_cache
from typing import List

def gemini_2_zero_free_palindromes(n: int) -> List[int]:
    """
    Returns a sorted list of integer palindromes (no zero digits) 
    whose digits sum exactly to `n`.
    """
    if n <= 0:
        return []

    @lru_cache(None)
    def build_palindromes(target_sum: int) -> List[str]:
        # If the sum overshoots, this path is invalid
        if target_sum < 0:
            return []
            
        # Base case for even-length palindromes: the sum is perfectly depleted,
        # so the innermost "middle" is simply an empty string.
        if target_sum == 0:
            return [""]
            
        results = []
        
        # Base case for odd-length palindromes: a single digit in the exact middle.
        # Because zero digits are disallowed, it must strictly be 1 through 9.
        if 1 <= target_sum <= 9:
            results.append(str(target_sum))
            
        # Recursive step: wrap a smaller palindrome with a pair of identical digits (1-9)
        # on the left and right, subtracting their combined value from the target sum.
        for digit in range(1, 10):
            remaining_sum = target_sum - (2 * digit)
            
            if remaining_sum >= 0:
                for inner_pal in build_palindromes(remaining_sum):
                    results.append(str(digit) + inner_pal + str(digit))
                    
        return results

    # Generate the string representations, convert to integers, and return sorted
    return sorted(int(p) for p in build_palindromes(n))

# GPT  

In [6]:
from collections.abc import Iterator


def _halves_with_sum(weights: tuple[int, ...], target: int) -> Iterator[str]:
    """Yield digit strings whose weighted digit sum is ``target``."""

    suffix_weight = [0] * (len(weights) + 1)
    for index in range(len(weights) - 1, -1, -1):
        suffix_weight[index] = suffix_weight[index + 1] + weights[index]

    digits: list[str] = []

    def visit(index: int, remaining: int) -> Iterator[str]:
        if index == len(weights):
            if remaining == 0:
                yield "".join(digits)
            return

        weight = weights[index]
        rest_weight = suffix_weight[index + 1]
        for digit in range(1, 10):
            after = remaining - weight * digit
            # Every remaining digit is in [1, 9].
            if rest_weight <= after <= 9 * rest_weight:
                digits.append(str(digit))
                yield from visit(index + 1, after)
                digits.pop()

    yield from visit(0, target)


def GPT_palindromes_with_digit_sum(n: int) -> Iterator[int]:
    """Yield positive zero-free decimal palindromes with digit sum ``n``.

    Results are yielded in increasing numeric order.  The sequence is finite:
    since every digit is at least one, a result can contain at most ``n``
    digits.
    """

    if n < 1:
        return

    for length in range(1, n + 1):
        paired_digits = length // 2
        weights = (2,) * paired_digits
        if length % 2:
            weights += (1,)

        minimum_sum = sum(weights)
        maximum_sum = 9 * minimum_sum
        if not minimum_sum <= n <= maximum_sum:
            continue

        for half in _halves_with_sum(weights, n):
            left = half[:paired_digits]
            if length % 2:
                palindrome = half + left[::-1]
            else:
                palindrome = half + half[::-1]
            yield int(palindrome)

In [7]:
from functools import cache

@cache
def GPT_2_palindromes(n):
    """Return all positive base-10 palindromes with no zero digits
    whose digits sum to n.
    """
    if n == 0:
        return ("",)  # empty palindrome, useful for constructing 11, 22, ...

    result = []

    # A one-digit palindrome.
    if 1 <= n <= 9:
        result.append(str(n))

    # Wrap a smaller palindrome with matching digits d ... d.
    # This adds 2*d to the digit sum.
    for d in range(1, 10):
        if 2 * d > n:
            break
        for middle in GPT_2_palindromes(n - 2 * d):
            result.append(str(d) + middle + str(d))

    return tuple(result)

# Claude 

In [8]:
def gen_digit_sequences(length, target):
    """
    Yield every list of `length` digits, each in 1..9, that sums to `target`.
    Uses feasibility pruning (min/max achievable sum) to avoid dead branches.
    """
    if length == 0:
        if target == 0:
            yield []
        return

    remaining_positions = length - 1
    min_possible = remaining_positions * 1
    max_possible = remaining_positions * 9

    for d in range(1, 10):
        rem = target - d
        if min_possible <= rem <= max_possible:
            for tail in gen_digit_sequences(remaining_positions, rem):
                yield [d] + tail


def claude_find_palindromes(n):
    """
    Return a sorted list of ints: all zero-free palindromes whose digits
    sum to n.
    """
    if n < 1:
        return []

    palindromes = []
    max_length = n  # each digit >= 1, so length can't exceed the target sum

    for L in range(1, max_length + 1):
        if L % 2 == 0:
            h = L // 2
            if n % 2 != 0:
                continue  # 2*sum(half) is always even
            target = n // 2
            for half in gen_digit_sequences(h, target):
                full_digits = half + half[::-1]
                palindromes.append(full_digits)
        else:
            h = (L + 1) // 2
            if h == 1:
                # single-digit palindrome: the digit itself must equal n
                if 1 <= n <= 9:
                    palindromes.append([n])
                continue
            # half = prefix (h-1 digits) + middle digit m
            # n = 2*sum(prefix) + m
            for m in range(1, 10):
                rem = n - m
                if rem < 0 or rem % 2 != 0:
                    continue
                prefix_target = rem // 2
                for prefix in gen_digit_sequences(h - 1, prefix_target):
                    half = prefix + [m]
                    full_digits = half + half[:-1][::-1]
                    palindromes.append(full_digits)

    numbers = sorted(int("".join(map(str, digits))) for digits in palindromes)
    return numbers

In [9]:
from functools import lru_cache

@lru_cache(maxsize=None)
def claude_2_palindromes_with_digit_sum(n):
    """Palindromic integers in base 10, with no zero digits,
    whose digits add up to n, as a sorted tuple."""
    if n < 1:
        return ()
    result = []
    if n <= 9:
        result.append(n)                        # single digit
    for d in range(1, min(9, n // 2) + 1):
        rest = n - 2 * d
        if rest == 0:
            result.append(11 * d)               # just "dd"
        else:
            for p in claude_2_palindromes_with_digit_sum(rest):   # smaller subproblem
                k = len(str(p))
                result.append(d * 10**(k + 1) + p * 10 + d)  # wrap: d p d
    return tuple(sorted(result))

# Kimi 

In [10]:
def kimi_palindromes_with_digit_sum(digit_sum, *, as_int=False):
    """
    Return every positive base-10 palindrome with no zero digits whose
    digit sum equals `digit_sum`.

    Returns a numerically sorted list of strings by default.
    Set as_int=True to return integers.
    """
    if not isinstance(digit_sum, int) or isinstance(digit_sum, bool):
        raise TypeError("digit_sum must be an integer")
    if digit_sum < 1:
        return []

    def compositions(total, parts):
        """Generate compositions of total into parts, each between 1 and 9."""
        if parts == 1:
            if 1 <= total <= 9:
                yield (total,)
            return

        lo = max(1, total - 9 * (parts - 1))
        hi = min(9, total - (parts - 1))

        for first in range(lo, hi + 1):
            for rest in compositions(total - first, parts - 1):
                yield (first,) + rest

    found = set()

    # Single-digit palindromes.
    if digit_sum <= 9:
        found.add(str(digit_sum))

    # Even-length palindromes: left + reverse(left).
    # These exist only when the digit sum is even.
    if digit_sum % 2 == 0:
        half_sum = digit_sum // 2

        for half_length in range(1, half_sum + 1):
            for left in compositions(half_sum, half_length):
                found.add("".join(map(str, left + left[::-1])))

    # Odd-length palindromes: left + center + reverse(left).
    # These must be checked for every digit sum because the center digit
    # can make the total digit sum either even or odd.
    for half_length in range(1, (digit_sum - 1) // 2 + 1):
        for center in range(1, 10):
            remaining = digit_sum - center

            if remaining < 0 or remaining % 2:
                continue

            left_sum = remaining // 2
            if not half_length <= left_sum <= 9 * half_length:
                continue

            for left in compositions(left_sum, half_length):
                digits = left + (center,) + left[::-1]
                found.add("".join(map(str, digits)))

    result = sorted(found, key=lambda x: (len(x), x))
    return [int(x) for x in result] if as_int else result

In [11]:
from functools import cache


@cache
def kimi_2_palindromes_with_digit_sum(n: int) -> tuple[str, ...]:
    """
    Return all palindromic strings composed of nonzero digits
    whose digits sum to n.

    The empty string is included only for n == 0 so that it can be
    used to construct two-digit palindromes such as 11 and 22.
    """
    if n == 0:
        return ("",)

    result = []

    # A one-digit palindrome.
    if 1 <= n <= 9:
        result.append(str(n))

    # Surround a smaller palindrome with equal nonzero digits.
    # The digit sum increases by 2*d.
    for d in range(1, 10):
        if 2 * d > n:
            break

        for middle in kimi_2_palindromes_with_digit_sum(n - 2 * d):
            result.append(f"{d}{middle}{d}")

    return tuple(result)

# Unit Tests and Timing Results

My function `test` runs a series of test cases for various values of *N* up to 42 (for which there are 2,007,494 valid palindromes) and returns the run time in seconds. I use the local function `f` for canonicalization:  an implementation might reasonably choose to produce ints instead of strings, and might return (or yield) the palindromes in any order, not necessarily lexicographic order. If the palindrome function has a cache, it is cleared at the start. 

In [12]:
import time

def test(palindrome_function) -> float:
    """Tests for the palindrome digit sum problem; return total run time."""

    if hasattr(palindrome_function, 'cache_clear'):
        palindrome_function.cache_clear()
    
    def f(N: int) -> list[str]: 
        """Canonicalize the output of palindrome_function(N)."""
        return sorted(map(str, palindrome_function(N)))

    start_time = time.perf_counter()

    # Tests for small values of N
    assert f(0) == [''] or f(0) == [] # Unspecified; either answer is acceptable
    assert f(1) == ['1']
    assert f(2) == ['11',                                                          '2']
    assert f(3) == ['111',                                                         '3']
    assert f(4) == ['1111',                      '121',     '22',                  '4']
    assert f(5) == ['11111',                     '131',    '212',                  '5']
    assert f(6) == ['111111',  '11211', '1221',  '141',   '2112',   '222',   '33', '6']
    assert f(7) == ['1111111', '11311', '12121', '151',  '21112',   '232',  '313', '7']

    # Try the function on some larger N and check the number of palindromes.
    for N, number in {38: 502_864, 40: 1_004_737, 42: 2_007_494}.items():
        assert len(f(N)) == number == len(f(N + 1))

    return time.perf_counter() - start_time # Total run time

Here we see the time each function takes to run the test suite. Note all assertions were valid for all functions.

In [13]:
functions = (palindromes_with_digit_sum,
             gemini_generate_palindromes,
             GPT_palindromes_with_digit_sum,
             claude_find_palindromes,
             kimi_palindromes_with_digit_sum,
             gemini_2_zero_free_palindromes,
             GPT_2_palindromes,
             claude_2_palindromes_with_digit_sum,
             kimi_2_palindromes_with_digit_sum)

for fn in functions:
    print(f'{test(fn):4.1f} sec {fn.__name__}')

 0.7 sec palindromes_with_digit_sum
 7.5 sec gemini_generate_palindromes
10.1 sec GPT_palindromes_with_digit_sum
16.4 sec claude_find_palindromes
14.6 sec kimi_palindromes_with_digit_sum
 2.5 sec gemini_2_zero_free_palindromes
 1.1 sec GPT_2_palindromes
 1.9 sec claude_2_palindromes_with_digit_sum
 1.0 sec kimi_2_palindromes_with_digit_sum
